# Config

### Load

In [ ]:
!pip install sentence-transformers==5.1.0     

In [2]:
!git config --global --add safe.directory /tmp/Repository/VRID_language_proyect

In [ ]:
import os

# Ruta a la que quieres mover el path
nueva_ruta = "/tmp/Repository/VRID_language_proyect/BERT"

# Cambiar el directorio actual
os.chdir(nueva_ruta)

# Confirmar que cambió
print("Directorio actual:", os.getcwd())

Directorio actual: /tmp/Repository/VRID_language_proyect/BERT


In [4]:
import pandas as pd
import os
from preprocess.preprocess import clean_text, check_deleted_expressions
from preprocess.translate import translator, gen_text_for_embedding, final_clean, detect_language
import time
import json
import numpy as np

### Functions

In [5]:
import io
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
from PIL import Image

#def register_confusion_matrix(y, preds):    
#    # Generar matriz de confusión normalizada
#    cm = confusion_matrix(y, preds, normalize="true")
 # Convertir el DataFrame a un array de numpy
def register_confusion_matrix(df_cm):
    cm = df_cm.to_numpy().astype(float)
    # Normalizar por filas (cada fila suma 1)
    cm = cm / cm.sum(axis=1, keepdims=True)
    # Crear la figura
    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(cm, interpolation="nearest", cmap=plt.cm.Blues)
    ax.set_title("Matriz de Confusión")
    fig.colorbar(im, ax=ax)

    # Agregar valores dentro de cada celda
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, f"{cm[i, j]:.3f}",
                    ha="center", va="center", color="black")

    ax.set_xlabel("Predicción")
    ax.set_ylabel("Real")
    plt.tight_layout()

    # Guardar en memoria como PNG
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=150, bbox_inches="tight")
    buf.seek(0)

    # Convertir a PIL.Image
    img = Image.open(buf)

    # Importante: cerrar figura para que no se muestre ni ocupe memoria
    plt.close(fig)

    return img


In [6]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
)


from sklearn.metrics import confusion_matrix

import numpy as np

import mlflow
import os
import git
import tempfile
import matplotlib.pyplot as plt

def metrics_lang(y, preds, lang_es):
    #Conversión en array
    y = np.array(y)
    preds = np.array(preds)
    lang_es = np.array(lang_es)

    # Seleccionar por máscara booleana
    y_es = y[lang_es] #Data originalmente en español
    preds_es = preds[lang_es]

    y_en = y[~lang_es] #Data originalmente en inglés
    preds_en = preds[~lang_es]
    
    #Computo de métricas
    f1_es = f1_score(y_es, preds_es, average="weighted")
    f1_en = f1_score(y_en, preds_en, average="weighted")
    cm_es = confusion_matrix(y_es, preds_es)
    cm_en = confusion_matrix(y_en, preds_en)

    return f1_es, f1_en, cm_es, cm_en

def eval_model(best_model, X_test, y_test, lang_es=None, mode = "binary"):
    results = {}
    preds = best_model.predict(X_test)
    cm = confusion_matrix(y_test, preds)
    #Métricas generales
    results = {
        'accuracy': accuracy_score(y_test, preds),
        'f1_macro': f1_score(y_test, preds, zero_division=0, average="macro"),
        'cm': cm,
    }
    # Métricas adicionales para clasificación binaria
    if mode == "binary":
        results.update({
            'precision': precision_score(y_test, preds, zero_division=0),
                'recall': recall_score(y_test, preds, zero_division=0)
            })
    # Métricas por idioma
    if lang_es is not None:
        f1_es, f1_en, cm_es, cm_en = metrics_lang(y_test, preds, lang_es)
        results.update({
            'f1_es': f1_es,
            'f1_en': f1_en,
            'cm_es': cm_es,
            'cm_en': cm_en
        })
    return results, preds

#MLflow logging helper function
def safe_log_metric(name, value):
    try:
        if isinstance(value, (list, tuple, np.ndarray)):
            if np.size(value) == 1:
                value = float(np.array(value).item())
            else:
                raise ValueError("Métrica con más de un valor.")
        else:
            value = float(value)
        mlflow.log_metric(name, value)
    except Exception as e:
        print(f"⚠️ No se pudo loggear {name}: {e}")

def log_artifact_generic(k, v):
    #Guardar artefactos csv y json
    with tempfile.TemporaryDirectory() as tmpdir:
        if isinstance(v, dict):
            # Guardar como JSON
            fname = os.path.join(tmpdir, f"{k}.json")
            with open(fname, "w", encoding="utf-8") as f:
                json.dump(v, f, indent=4, ensure_ascii=False)

        elif isinstance(v, pd.DataFrame):
            # Guardar como CSV
            fname = os.path.join(tmpdir, f"{k}.csv")
            v.to_csv(fname, index=False)

        elif isinstance(v, str) and os.path.exists(v):
            # Si ya es una ruta válida
            fname = v
        
        elif isinstance(v, Image.Image):
            fname = os.path.join(tmpdir, f"{k}.png")
            v.save(fname, format="PNG")

        else:
            # Si no sabes qué es, lo guardamos como string plano
            fname = os.path.join(tmpdir, f"{k}.txt")
            with open(fname, "w", encoding="utf-8") as f:
                f.write(str(v))

        # Log en MLflow
        mlflow.log_artifact(fname, artifact_path=k)

def mlflow_ckeckpoint(exp_info, results_val, models_dicc, X_test, y_test, df_test, save_preds=None, lang_es=None, extra_parms=None, extra_artifacts = None, mode="server", mode_classification="binary"):
    
    if mode == "server":
        # Set backend store
        mlflow.set_tracking_uri("http://mlflow-server:5000")
        tracking_uri = mlflow.get_tracking_uri()
        print("Current tracking uri: {}".format(tracking_uri)) 
    
    elif mode == "local": 
        # Set backend store
        mlflow.set_tracking_uri(exp_info["tracking_path"])
        tracking_uri = mlflow.get_tracking_uri()
        print("Current tracking uri: {}".format(tracking_uri)) 

        # Verificar si existe experimento, si no crearlo
        experiment = mlflow.get_experiment_by_name(exp_info["exp_name"])

        if experiment is None:
            exp_id = mlflow.create_experiment(
                exp_info["exp_name"],
                artifact_location=exp_info["artifact_path"]
            )
            print(f"Experimento creado con ID: {exp_id}")
        else:
            exp_id = experiment.experiment_id
            print(f"Experimento ya existe con ID: {exp_id}")
    
    else: 
        print("Especificar modo de almacenamiento")
        return 0

    # Define el experimento (lo crea si no existe)
    mlflow.set_experiment(exp_info["exp_name"])
    
    # Obtener commit actual
    repo = git.Repo(search_parent_directories=True)
    commit_hash = repo.head.object.hexsha

    for model_name, metrics in results_val.items():
        model = models_dicc[model_name]

        with mlflow.start_run(run_name=model_name):
            print(f"📝 Registrando modelo en MLflow: {model_name}")

            # Hiperparámetros
            try:
                mlflow.log_params(model.get_params())
            except:
                print(f"⚠️ No se pudieron loggear los hiperparámetros para {model_name}")

            #Parámetros adicionales, se pueden añadir con un diccionario
            if extra_parms is not None:
                for k, v in extra_parms.items():
                    mlflow.log_param(k, v)

            #Artefactos adicionales, pueden ser json, csv, txt, dataframe
            if extra_artifacts is not None:
                for k, v in extra_artifacts.items():
                    log_artifact_generic(k, v)

            # Métricas de validación
            for k, v in metrics.items():
                safe_log_metric(f"val_{k}", v)

            #Predicciones y resultados de test
            results_test, preds = eval_model(model, X_test, y_test, lang_es, mode = mode_classification)

            # Generar y guardar matriz de confusión
            #cm_img = register_confusion_matrix(y_test, preds)
            #log_artifact_generic("cm_normalized_img", cm_img)

            #Guardar predicciones
            if save_preds is not None:
                df_test["y_true"]=y_test
                df_test["preds"]=preds
                log_artifact_generic("df_test", df_test)

            # Guardar métricas de test
            for k, v in results_test.items():
                if k.startswith("cm"):
                    # Guardar confusion matrix (o similar) como artefacto
                    # Guardar como CSV temporal
                    v = pd.DataFrame(v)
                    log_artifact_generic(k, v)
                    #Guardar imagen
                    cm_img = register_confusion_matrix(v)
                    log_artifact_generic(f"{k}_img", cm_img)
                    
                else:
                    # Guardar métrica numérica
                    safe_log_metric(f"test_{k}", v)
            
            #Guardar commit de git
            mlflow.log_param("git_commit", commit_hash)
                    
            # Guardar modelo
            mlflow.sklearn.log_model(model, artifact_path = "model", input_example=X_test[:5])
    

# 1) Preprocesamiento de los datos


In [ ]:
# 1) Cargar datos
path = "/tmp/data"
path_analytics = "/tmp/analytics"
filePATH = os.path.join(path, "data_concatenada.xlsx")
df = pd.read_excel(filePATH,
                   usecols=["Código VRID", "Título", "Resumen", "Keywords", "Interdisciplinario", "Transdisciplinario", "Facultad del Proyecto",
                            "Depto Persona"]) \
       .fillna("")

# 2) Guardar qué secuencias de palabras del resumen serán eliminadas al aplicar get_expressions_to_delete()
list_texts = df["Resumen"].to_list()
df_deleted = check_deleted_expressions(list_texts)
savepath=os.path.join(path_analytics, "deleted_re.xlsx")
df_deleted.to_excel(savepath, index=False)

# 3) Preprocesar los datos
#Columnas que se van a preprocesar
#Nombre fila seleccionada:Columna que se creará para guardar resultado
cols = {
    "Título": "Titulo_trad",
    "Resumen": "Resumen_trad",
    "Keywords": "keywords_trad",
    "Facultad del Proyecto": "Facultad_del_Proyecto_trad",
    "Depto Persona": "Depto_Persona_trad",
}
#Preprocesamiento de datos
df[list(cols.values())] = df[list(cols.keys())].applymap(clean_text)
savepath=os.path.join(path, "data_clean.xlsx")
df.to_excel(savepath, index=False)

# 2) Traducción del texto

In [ ]:
#Crear columna de registro de idioma: 
# True: Texto en español, False: Texto en inglés
df["Español"]=detect_language(df["Resumen_trad"])

In [ ]:
from transformers import MarianMTModel, MarianTokenizer

#1. Cargar modelo de traducción
model_name = "Helsinki-NLP/opus-mt-es-en"
tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name)
trans = translator(model, tokenizer)

#Columnas que se van a traducir
#Nombre fila seleccionada:Columna que se creará para guardar resultado
cols = {
    "Titulo_trad": "Titulo_trad",
    "Resumen_trad": "Resumen_trad",
    "keywords_trad": "keywords_trad",
    "Facultad_del_Proyecto_trad": "Facultad_del_Proyecto_trad",
    "Depto_Persona_trad": "Depto_Persona_trad",
}

#2. Traducción de columnas
#####Estoy trabajando en mejorar esta parte para que sea más rápida con paralelización por batches
start = time.time()
for src, dst in cols.items():
    df[dst] = trans.translate_parallel(df[src].to_list(), batch_size=8)
end = time.time()


#3.Guardado de resultados
savepath=os.path.join(path, "data_translated.xlsx")
df.to_excel(savepath, index=False)

print(f"Tiempo total de traducción: {end - start:.2f} segundos")

In [ ]:
#3. Selección de columnas que se utilizarán en clasificador y concatenación
# Última limpieza antes de generar concatenación
cols = ["Titulo_trad", "keywords_trad", "Resumen_trad", "Facultad_del_Proyecto_trad", "Depto_Persona_trad"]
for col in cols:
    df[col] = df[col].apply(final_clean)

# Guardado de resultados
savepath=os.path.join(path, "data_translated_concat.xlsx")
df.to_excel(savepath, index=False)
savepath=os.path.join(path, "data_translated_concat.csv")
df.to_csv(savepath, index=False, encoding="utf-8-sig")

df.head()

# 3) Split dataset

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split, StratifiedKFold
from utils.dataset import to_serializable
import json

path = "/tmp/data"
filepath=os.path.join(path, "data_translated_concat.csv")
df = pd.read_csv(filepath)

df = df[df["Interdisciplinario"] != "INDEFINIDO"]
le = LabelEncoder()
df["labels"] = le.fit_transform(df["Interdisciplinario"])

# Convertir a arrays
ids = df["Código VRID"].to_numpy()
labels = df["labels"].to_numpy()

# Train/Test split (ids y labels en paralelo)
idx_train, idx_test, y_train, y_test = train_test_split(
    ids,
    labels,
    test_size=0.2,
    random_state=7,
    stratify=labels
)

# Crear folds sobre train
skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=7)

folds = []
for fold, (train_pos, val_pos) in enumerate(skf.split(idx_train, y_train)):
    train_ids = idx_train[train_pos]   # array de IDs
    val_ids = idx_train[val_pos]       # array de IDs
    folds.append(val_ids)

print("Test size:", len(idx_test))
print("Fold 0 - Val size:", len(folds[0]))

#Guardar index en diccionario
dataset_index = {
    "Train": idx_train,
    "Test": idx_test,
    "kfolds": folds 
}
filepath=os.path.join(path, "train_test_ids_3folds.json")

# Guardar
with open(filepath, "w", encoding="utf-8") as f:
    json.dump(dataset_index, f, default=to_serializable, indent=2, ensure_ascii=False)


Test size: 193
Fold 0 - Val size: 257


# 4) TF ID feature extractor

## Train

In [ ]:
#Ruta de lectura
path = "/tmp/data"

#Lectura de index de separacion de conjuntos train/test
filepath=os.path.join(path, "train_test_ids_3folds.json")
with open(filepath, "r", encoding="utf-8") as f:
    dataset_index = json.load(f)

#Lectura de data
filepath=os.path.join(path, "data_translated_concat.csv")
df = pd.read_csv(filepath)

#Prueba con solo textos traducidos
#df = df[df["Español"]==False]

In [8]:
from sklearn.preprocessing import LabelEncoder
from utils.dataset import gen_dataset_select_cols
from models.TIFD import gen_TFID_vectors
import numpy as np

#Columnas a seleccionar para clasificación
cols = ["Titulo_trad", "keywords_trad", "Resumen_trad"]

#Lectura de codigos VRID Test
codes_test = dataset_index["Test"]
X_test, y_test, df_test= gen_dataset_select_cols(codes_test, df, cols = cols)

#Lectura de codigos VRID Train
codes_train = dataset_index["kfolds"]
codes_train = np.array([i for fold in codes_train for i in fold])
X_train, y_train, df_train = gen_dataset_select_cols(codes_train, df, cols = cols)
df_decode = df_train[["idx", "Código VRID"]]

# Codificación de labels
le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)

#Creacion de vectores TFID
X_train, X_test = gen_TFID_vectors(X_train, X_test)
print(X_train.shape, X_test.shape)

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


(771, 16723) (193, 16723)


In [ ]:
from pipelines.ML_pipeline_skp import get_est_params_dict, run_bayesian_pipeline, select_best_model, #eval_model
from utils.dataset import CvCustom
from collections import Counter

# 1. Elegir modelos a probar
model_keys = [
    'LogisticRegression',
    #'RandomForestClassifier',
    #'XGBClassifier',
    #'SVC',
]

# 2. Obtener el diccionario de modelos y parámetros
est_params_dict = get_est_params_dict(model_keys)
print("📊 train:", Counter(y_train))
print("📊 test:", Counter(y_test))

# 3. Ejecutar entrenamiento, validación y test con tus funciones
n_iter=20
sample_weight_On=True
scoring='f1_macro'
results_val, models_dicc = run_bayesian_pipeline(est_params_dict, X_train, y_train, scoring=scoring, cv_function=CvCustom(df_decode), 
                                                 n_iter=n_iter, sample_weight_On = sample_weight_On)

best_model = select_best_model(results_val, models_dicc)

# 4. Mostrar resultados
print("\n🔍 Validación:")
for model, metrics in results_val.items():
    print(f"{model}: {metrics}")

📊 train: Counter({1: 444, 0: 327})
📊 test: Counter({1: 111, 0: 82})
(771,)
LogisticRegression
Compute sw


/usr/local/lib/python3.10/dist-packages/skopt/optimizer/optimizer.py:517: UserWarning: The objective has been evaluated at point [10, 'l2', 'lbfgs'] before, using random point [0.1, 'l2', 'lbfgs']
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/skopt/optimizer/optimizer.py:517: UserWarning: The objective has been evaluated at point [0.1, 'l2', 'lbfgs'] before, using random point [1.0, 'l2', 'lbfgs']
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/skopt/optimizer/optimizer.py:517: UserWarning: The objective has been evaluated at point [1.0, 'l2', 'lbfgs'] before, using random point [0.1, 'l2', 'lbfgs']
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/skopt/optimizer/optimizer.py:517: UserWarning: The objective has been evaluated at point [1.0, 'l2', 'lbfgs'] before, using random point [10, 'l2', 'lbfgs']
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/skopt/optimizer/optimizer.py:517: UserWarning: The objective has been evaluated at point [1.0, 'l2', '


🔍 Validación:
LogisticRegression: {'mean_test_score': 0.63, 'std_test_score': 0.01}


In [10]:
# Métricas por idioma
lang_es = df_test["Español"]
for name, model in models_dicc.items():
    print(name)
    results=eval_model(model, X_test, y_test, lang_es)
    print(results)

LogisticRegression
{'accuracy': 0.689119170984456, 'f1_macro': 0.6838138925294889, 'cm': array([[54, 28],
       [32, 79]]), 'f1_es': 0.6772899073139264, 'f1_en': 0.7024084418013649, 'cm_es': array([[21, 19],
       [18, 57]]), 'cm_en': array([[33,  9],
       [14, 22]]), 'precision': 0.7383177570093458, 'recall': 0.7117117117117117}


## Save

In [20]:
exp_info = {
    'exp_name': "test_MLflow_local",
}

extra_parms = {
    "vectorization_model": "TF-IDF",
    "n_iter": n_iter,
    "sample_weight_On": sample_weight_On,
    "scoring": scoring,
    "cols": cols
}

extra_artifacts = {
    "dataset_data_splits": dataset_index,
}

mlflow_ckeckpoint(exp_info, results_val, models_dicc, X_test, y_test, df_test, save_preds=True, lang_es=lang_es, extra_parms=extra_parms, extra_artifacts=extra_artifacts, mode="server", mode_classification="binary")

Current tracking uri: http://mlflow-server:5000
📝 Registrando modelo en MLflow: LogisticRegression


2025/09/13 23:50:57 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run LogisticRegression at: http://mlflow-server:5000/#/experiments/17/runs/7170cd73c5614a3786fdb7a91960f2f5
🧪 View experiment at: http://mlflow-server:5000/#/experiments/17


# 5) SPECTER model

### Train

In [7]:
#Ruta de lectura
path = "/tmp/data"

#Lectura de index de separacion de conjuntos train/test
filepath=os.path.join(path, "train_test_ids_3folds.json")
with open(filepath, "r", encoding="utf-8") as f:
    dataset_index = json.load(f)

#Lectura de data
filepath=os.path.join(path, "data_translated_concat.csv")
df = pd.read_csv(filepath)

In [8]:
from utils.dataset import gen_dataset_select_cols
from sklearn.preprocessing import LabelEncoder
from models.specter import embed_texts
import numpy as np


#Columnas a seleccionar para clasificación
cols = ["Titulo_trad", "keywords_trad", "Resumen_trad"]
element_names=["Title:", "keywords:", "abstract:"]

#Lectura de codigos VRID Test
codes_test = dataset_index["Test"]
X_test, y_test, df_test= gen_dataset_select_cols(codes_test, df, cols = cols, element_names=element_names)

#Lectura de codigos VRID Train
codes_train = dataset_index["kfolds"]
codes_train = np.array([i for fold in codes_train for i in fold])
X_train, y_train, df_train = gen_dataset_select_cols(codes_train, df, cols = cols)
df_decode = df_train[["idx", "Código VRID"]]

# Codificación de labels
le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)

# Calcular embeddings
# Parámetros para cargar modelo
BASE_MODEL = "allenai/specter2_base"
ADAPTER_NAME="allenai/specter2_classification"
X_train = embed_texts(X_train, BASE_MODEL, ADAPTER_NAME)
X_test = embed_texts(X_test, BASE_MODEL, ADAPTER_NAME)

print(X_train.shape, X_test.shape)

/usr/local/lib/python3.10/dist-packages/torch/_utils.py:830: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

There are adapters available but none are activated for the forward pass.


Modelo SPECTER2 cargado correctamente.


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

There are adapters available but none are activated for the forward pass.


Modelo SPECTER2 cargado correctamente.
(771, 768) (193, 768)


In [61]:
from pipelines.ML_pipeline_skp import get_est_params_dict, run_bayesian_pipeline, select_best_model#, eval_model
from utils.dataset import CvCustom
from collections import Counter

# 1. Elegir modelos a probar
model_keys = [
    'LogisticRegression',
    'RandomForestClassifier',
    'XGBClassifier',
    'SVC',
]

# 2. Obtener el diccionario de modelos y parámetros
est_params_dict = get_est_params_dict(model_keys)
print("📊 train:", Counter(y_train))
print("📊 test:", Counter(y_test))

# 3. Ejecutar entrenamiento, validación y test con tus funciones
n_iter=20
sample_weight_On=True
scoring="f1_macro"
results_val, models_dicc = run_bayesian_pipeline(est_params_dict, X_train, y_train, scoring=scoring, cv_function=CvCustom(df_decode), 
                                                 n_iter=n_iter, sample_weight_On = sample_weight_On)

best_model = select_best_model(results_val, models_dicc)

# 4. Mostrar resultados
print("\n🔍 Validación:")
for model, metrics in results_val.items():
    print(f"{model}: {metrics}")

📊 train: Counter({1: 444, 0: 327})
📊 test: Counter({1: 111, 0: 82})
(771,)
LogisticRegression
Compute sw


/usr/local/lib/python3.10/dist-packages/skopt/optimizer/optimizer.py:517: UserWarning: The objective has been evaluated at point [10, 'l2', 'lbfgs'] before, using random point [0.1, 'l2', 'lbfgs']
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/skopt/optimizer/optimizer.py:517: UserWarning: The objective has been evaluated at point [10, 'l2', 'lbfgs'] before, using random point [1.0, 'l2', 'lbfgs']
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/skopt/optimizer/optimizer.py:517: UserWarning: The objective has been evaluated at point [1.0, 'l2', 'lbfgs'] before, using random point [0.1, 'l2', 'lbfgs']
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/skopt/optimizer/optimizer.py:517: UserWarning: The objective has been evaluated at point [10, 'l2', 'lbfgs'] before, using random point [10, 'l2', 'lbfgs']
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/skopt/optimizer/optimizer.py:517: UserWarning: The objective has been evaluated at point [10, 'l2', 'lbf

RandomForestClassifier
Compute sw
XGBClassifier
Compute sw
SVC
Compute sw

🔍 Validación:
LogisticRegression: {'mean_test_score': 0.64, 'std_test_score': 0.0}
RandomForestClassifier: {'mean_test_score': 0.63, 'std_test_score': 0.02}
XGBClassifier: {'mean_test_score': 0.63, 'std_test_score': 0.04}
SVC: {'mean_test_score': 0.65, 'std_test_score': 0.0}


In [62]:
# Métricas por idioma
lang_es = df_test["Español"]
for name, model in models_dicc.items():
    print(name)
    results=eval_model(model, X_test, y_test, lang_es)
    print(results)

LogisticRegression
{'accuracy': 0.6580310880829016, 'f1_macro': 0.6464023984010661, 'cm': array([[46, 36],
       [30, 81]]), 'f1_es': 0.6309720505147373, 'f1_en': 0.6780455925883212, 'cm_es': array([[15, 25],
       [16, 59]]), 'cm_en': array([[31, 11],
       [14, 22]]), 'precision': 0.6923076923076923, 'recall': 0.7297297297297297}
RandomForestClassifier
{'accuracy': 0.6787564766839378, 'f1_macro': 0.6702491181657848, 'cm': array([[50, 32],
       [30, 81]]), 'f1_es': 0.6475378168742013, 'f1_en': 0.7171990799897776, 'cm_es': array([[18, 22],
       [18, 57]]), 'cm_en': array([[32, 10],
       [12, 24]]), 'precision': 0.7168141592920354, 'recall': 0.7297297297297297}
XGBClassifier
{'accuracy': 0.6528497409326425, 'f1_macro': 0.627874183429739, 'cm': array([[38, 44],
       [23, 88]]), 'f1_es': 0.6486230277322537, 'f1_en': 0.627777286005134, 'cm_es': array([[14, 26],
       [12, 63]]), 'cm_en': array([[24, 18],
       [11, 25]]), 'precision': 0.6666666666666666, 'recall': 0.7927927927

## Save

In [68]:
exp_info = {
    'exp_name': "Bayesiansearchcv_Interdiciplinario_all_results",
}

extra_parms = {
    "vectorization_model": "specter2_classification",
    "n_iter": n_iter,
    "sample_weight_On": sample_weight_On,
    "scoring": scoring,
    "cols": cols
}

extra_artifacts = {
    "dataset_data_splits": dataset_index,
}

mlflow_ckeckpoint(exp_info, results_val, models_dicc, X_test, y_test, df_test, save_preds=True, lang_es=lang_es, extra_parms=extra_parms, extra_artifacts=extra_artifacts, mode="server", mode_classification="binary")

Current tracking uri: http://mlflow-server:5000
📝 Registrando modelo en MLflow: LogisticRegression


2025/09/12 17:43:30 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run LogisticRegression at: http://mlflow-server:5000/#/experiments/16/runs/d221e7b187974f38abfc41d2e4993466
🧪 View experiment at: http://mlflow-server:5000/#/experiments/16
📝 Registrando modelo en MLflow: RandomForestClassifier


2025/09/12 17:43:35 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run RandomForestClassifier at: http://mlflow-server:5000/#/experiments/16/runs/5b1f51e8f70d4953ac5560b88b99542e
🧪 View experiment at: http://mlflow-server:5000/#/experiments/16
📝 Registrando modelo en MLflow: XGBClassifier


2025/09/12 17:43:40 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run XGBClassifier at: http://mlflow-server:5000/#/experiments/16/runs/2f4f28f9e9f5404c96c166d83f9a26e1
🧪 View experiment at: http://mlflow-server:5000/#/experiments/16
📝 Registrando modelo en MLflow: SVC


2025/09/12 17:43:45 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run SVC at: http://mlflow-server:5000/#/experiments/16/runs/554a8de5ca1a462081e9deeb08931dec
🧪 View experiment at: http://mlflow-server:5000/#/experiments/16


# 6) all-roberta-large-v1

## Functions

In [82]:
from transformers import RobertaTokenizerFast, RobertaModel
import torch
import warnings
from tqdm import tqdm
warnings.filterwarnings("ignore", message="Some weights of the model.*were not initialized.*")

def roberta_encoder_batch(texts, batch_size=8, max_length=512):

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model_name = "roberta-large"
    tokenizer = RobertaTokenizerFast.from_pretrained(model_name)
    model = RobertaModel.from_pretrained(model_name).to(device)

    model.eval()  # desactiva dropout
    embeddings = []

    # recorrer en lotes de batch_size
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i+batch_size]

        # tokenización por lote
        inputs = tokenizer(
            batch_texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=max_length
        ).to(device) 

        with torch.no_grad():
            outputs = model(**inputs)

        # embeddings del token <s> ([CLS]) para cada texto del batch
        cls_embeddings = outputs.last_hidden_state[:, 0, :]  # (batch, hidden_dim)
        embeddings.append(cls_embeddings.cpu().numpy())

    # concatenar todos los batches
    return np.vstack(embeddings)  # (n_texts, hidden_dim)

## Train

In [83]:
#Ruta de lectura
path = "/tmp/data"

#Lectura de index de separacion de conjuntos train/test
filepath=os.path.join(path, "train_test_ids_3folds.json")
with open(filepath, "r", encoding="utf-8") as f:
    dataset_index = json.load(f)

#Lectura de data
filepath=os.path.join(path, "data_translated_concat.csv")
df = pd.read_csv(filepath)

In [84]:
#del gen_dataset
from utils.dataset import gen_dataset_select_cols
import numpy as np
from sklearn.preprocessing import LabelEncoder

#Columnas a seleccionar para clasificación
cols = ["Titulo_trad", "keywords_trad", "Resumen_trad"]
element_names=["Title:", "keywords:", "abstract:"]

#Lectura de codigos VRID Test
codes_test = dataset_index["Test"]
X_test, y_test, df_test= gen_dataset_select_cols(codes_test, df, cols = cols, element_names=element_names)

#Lectura de codigos VRID Train
codes_train = dataset_index["kfolds"]
codes_train = np.array([i for fold in codes_train for i in fold])
X_train, y_train, df_train = gen_dataset_select_cols(codes_train, df, cols = cols)
df_decode = df_train[["idx", "Código VRID"]]

# Codificación de labels
le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)

# Calcular embeddings
X_train = roberta_encoder_batch(X_train)
X_test = roberta_encoder_batch(X_test)
print(X_train.shape, X_test.shape)

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


(771, 1024) (193, 1024)


In [89]:
from pipelines.ML_pipeline_skp import get_est_params_dict, run_bayesian_pipeline, select_best_model#, eval_model
from utils.dataset import CvCustom
from collections import Counter

# 1. Elegir modelos a probar
model_keys = [
    'LogisticRegression',
    'RandomForestClassifier',
    'XGBClassifier',
    'SVC',
]

# 2. Obtener el diccionario de modelos y parámetros
est_params_dict = get_est_params_dict(model_keys)
print("📊 train:", Counter(y_train))
print("📊 test:", Counter(y_test))

# 3. Ejecutar entrenamiento, validación y test con tus funciones
n_iter=20
sample_weight_On=True
scoring="f1_macro"
results_val, models_dicc = run_bayesian_pipeline(est_params_dict, X_train, y_train, scoring=scoring, cv_function=CvCustom(df_decode), 
                                                 n_iter=n_iter, sample_weight_On = sample_weight_On)

best_model = select_best_model(results_val, models_dicc)

# 4. Mostrar resultados
print("\n🔍 Validación:")
for model, metrics in results_val.items():
    print(f"{model}: {metrics}")

📊 train: Counter({1: 444, 0: 327})
📊 test: Counter({1: 111, 0: 82})
(771,)
LogisticRegression
Compute sw


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

RandomForestClassifier
Compute sw
XGBClassifier
Compute sw
SVC
Compute sw

🔍 Validación:
LogisticRegression: {'mean_test_score': 0.61, 'std_test_score': 0.01}
RandomForestClassifier: {'mean_test_score': 0.61, 'std_test_score': 0.02}
XGBClassifier: {'mean_test_score': 0.62, 'std_test_score': 0.03}
SVC: {'mean_test_score': 0.6, 'std_test_score': 0.01}


In [90]:
# Métricas por idioma
lang_es = df_test["Español"]
for name, model in models_dicc.items():
    print(name)
    results, preds=eval_model(model, X_test, y_test, lang_es)
    print(results)

LogisticRegression
{'accuracy': 0.6476683937823834, 'f1_macro': 0.6434470767224516, 'cm': array([[52, 30],
       [38, 73]]), 'precision': 0.7087378640776699, 'recall': 0.6576576576576577, 'f1_es': 0.6772899073139264, 'f1_en': 0.5934065934065934, 'cm_es': array([[21, 19],
       [18, 57]]), 'cm_en': array([[31, 11],
       [20, 16]])}
RandomForestClassifier
{'accuracy': 0.6321243523316062, 'f1_macro': 0.6203418945501897, 'cm': array([[44, 38],
       [33, 78]]), 'precision': 0.6724137931034483, 'recall': 0.7027027027027027, 'f1_es': 0.6414924576902098, 'f1_en': 0.5819397993311036, 'cm_es': array([[14, 26],
       [13, 62]]), 'cm_en': array([[30, 12],
       [20, 16]])}
XGBClassifier
{'accuracy': 0.6528497409326425, 'f1_macro': 0.6388493227202905, 'cm': array([[44, 38],
       [29, 82]]), 'precision': 0.6833333333333333, 'recall': 0.7387387387387387, 'f1_es': 0.6878449790025153, 'f1_en': 0.5902844683332489, 'cm_es': array([[19, 21],
       [14, 61]]), 'cm_en': array([[25, 17],
       [1

## Save

In [93]:
exp_info = {
    'exp_name': "Bayesiansearchcv_Interdiciplinario_all_results",
}

extra_parms = {
    "vectorization_model": "all-roberta-large-v1",
    "n_iter": n_iter,
    "sample_weight_On": sample_weight_On,
    "scoring": scoring,
    "cols": cols
}

extra_artifacts = {
    "dataset_data_splits": dataset_index,
}

mlflow_ckeckpoint(exp_info, results_val, models_dicc, X_test, y_test, df_test, save_preds=True, lang_es=lang_es, extra_parms=extra_parms, extra_artifacts=extra_artifacts, mode="server", mode_classification="binary")

Current tracking uri: http://mlflow-server:5000
📝 Registrando modelo en MLflow: LogisticRegression


2025/09/12 18:07:49 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run LogisticRegression at: http://mlflow-server:5000/#/experiments/16/runs/0d5aa1be272d41508268d8c084cc64f8
🧪 View experiment at: http://mlflow-server:5000/#/experiments/16
📝 Registrando modelo en MLflow: RandomForestClassifier


2025/09/12 18:07:54 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run RandomForestClassifier at: http://mlflow-server:5000/#/experiments/16/runs/f8f5b0ba70f04c9f96d77653c600aed2
🧪 View experiment at: http://mlflow-server:5000/#/experiments/16
📝 Registrando modelo en MLflow: XGBClassifier


2025/09/12 18:07:59 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run XGBClassifier at: http://mlflow-server:5000/#/experiments/16/runs/c26a0ffa96684ca9add17720c250fe79
🧪 View experiment at: http://mlflow-server:5000/#/experiments/16
📝 Registrando modelo en MLflow: SVC


2025/09/12 18:08:04 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run SVC at: http://mlflow-server:5000/#/experiments/16/runs/e3e83e0f5f44440893a617fd4231dc0f
🧪 View experiment at: http://mlflow-server:5000/#/experiments/16


# 7) all-mpnet-base-v2 

## Train

In [72]:
#Ruta de lectura
path = "/tmp/data"

#Lectura de index de separacion de conjuntos train/test
filepath=os.path.join(path, "train_test_ids_3folds.json")
with open(filepath, "r", encoding="utf-8") as f:
    dataset_index = json.load(f)

#Lectura de data
filepath=os.path.join(path, "data_translated_concat.csv")
df = pd.read_csv(filepath)

In [73]:
#del gen_dataset
from utils.dataset import gen_dataset_select_cols
from sklearn.preprocessing import LabelEncoder
import numpy as np

#Columnas a seleccionar para clasificación
cols = ["Titulo_trad", "keywords_trad", "Resumen_trad"]
element_names=["Title:", "keywords:", "abstract:"]

#Lectura de codigos 
codes_test = dataset_index["Test"]
X_test, y_test, df_test= gen_dataset_select_cols(codes_test, df, cols = cols, element_names=element_names)

#Lectura de codigos 
codes_train = dataset_index["kfolds"]
codes_train = np.array([i for fold in codes_train for i in fold])
X_train, y_train, df_train = gen_dataset_select_cols(codes_train, df, cols = cols)
df_decode = df_train[["idx", "Código VRID"]]

# Codificación de labels
le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)

In [74]:
from models.sentence_transformers_models import encoder_sentence_transformers

model_name="all-mpnet-base-v2"
X_train = encoder_sentence_transformers(model_name, X_train)
X_test = encoder_sentence_transformers(model_name, X_test)

print(X_train.shape, X_test.shape)

Batches:   0%|          | 0/97 [00:00<?, ?it/s]

Batches:   0%|          | 0/25 [00:00<?, ?it/s]

(771, 768) (193, 768)


In [77]:
from pipelines.ML_pipeline_skp import get_est_params_dict, run_bayesian_pipeline, select_best_model #, eval_model
from utils.dataset import CvCustom
from collections import Counter

# 1. Elegir modelos a probar
model_keys = [
    'LogisticRegression',
    'RandomForestClassifier',
    'XGBClassifier',
    'SVC',
]

# 2. Obtener el diccionario de modelos y parámetros
est_params_dict = get_est_params_dict(model_keys)
print("📊 train:", Counter(y_train))
print("📊 test:", Counter(y_test))

# 3. Ejecutar entrenamiento, validación y test con tus funciones
n_iter=20
sample_weight_On=True
scoring="f1_macro"
results_val, models_dicc = run_bayesian_pipeline(est_params_dict, X_train, y_train, scoring=scoring, cv_function=CvCustom(df_decode), 
                                                 n_iter=n_iter, sample_weight_On = sample_weight_On)

best_model = select_best_model(results_val, models_dicc)

# 4. Mostrar resultados
print("\n🔍 Validación:")
for model, metrics in results_val.items():
    print(f"{model}: {metrics}")

📊 train: Counter({1: 444, 0: 327})
📊 test: Counter({1: 111, 0: 82})
(771,)
LogisticRegression
Compute sw


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

RandomForestClassifier
Compute sw
XGBClassifier
Compute sw
SVC
Compute sw

🔍 Validación:
LogisticRegression: {'mean_test_score': 0.62, 'std_test_score': 0.01}
RandomForestClassifier: {'mean_test_score': 0.64, 'std_test_score': 0.01}
XGBClassifier: {'mean_test_score': 0.61, 'std_test_score': 0.02}
SVC: {'mean_test_score': 0.62, 'std_test_score': 0.01}


In [80]:
# Métricas por idioma
lang_es = df_test["Español"]
for name, model in models_dicc.items():
    print(name)
    results, preds=eval_model(model, X_test, y_test, lang_es)
    print(results)

LogisticRegression
{'accuracy': 0.6269430051813472, 'f1_macro': 0.624025974025974, 'cm': array([[52, 30],
       [42, 69]]), 'precision': 0.696969696969697, 'recall': 0.6216216216216216, 'f1_es': 0.5772655840754322, 'f1_en': 0.6983339241403759, 'cm_es': array([[17, 23],
       [26, 49]]), 'cm_en': array([[35,  7],
       [16, 20]])}
RandomForestClassifier
{'accuracy': 0.6787564766839378, 'f1_macro': 0.6723439211391018, 'cm': array([[52, 30],
       [32, 79]]), 'precision': 0.7247706422018348, 'recall': 0.7117117117117117, 'f1_es': 0.6299147077179114, 'f1_en': 0.7404817404817405, 'cm_es': array([[17, 23],
       [19, 56]]), 'cm_en': array([[35,  7],
       [13, 23]])}
XGBClassifier
{'accuracy': 0.689119170984456, 'f1_macro': 0.6758844603672189, 'cm': array([[47, 35],
       [25, 86]]), 'precision': 0.7107438016528925, 'recall': 0.7747747747747747, 'f1_es': 0.6817895400126024, 'f1_en': 0.6791154164807852, 'cm_es': array([[17, 23],
       [12, 63]]), 'cm_en': array([[30, 12],
       [13, 

## Save

In [81]:
exp_info = {
    'exp_name': "Bayesiansearchcv_Interdiciplinario_all_results",
}

extra_parms = {
    "vectorization_model": "SentenceTransformers_all-mpnet-base-v2",
    "n_iter": n_iter,
    "sample_weight_On": sample_weight_On,
    "scoring": scoring,
    "cols": cols
}

extra_artifacts = {
    "dataset_data_splits": dataset_index,
}

mlflow_ckeckpoint(exp_info, results_val, models_dicc, X_test, y_test, df_test, save_preds=True, lang_es=lang_es, extra_parms=extra_parms, extra_artifacts=extra_artifacts, mode="server", mode_classification="binary")

Current tracking uri: http://mlflow-server:5000
📝 Registrando modelo en MLflow: LogisticRegression


2025/09/12 17:52:56 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run LogisticRegression at: http://mlflow-server:5000/#/experiments/16/runs/1bb4624ba1224af9bab50859e34f2f96
🧪 View experiment at: http://mlflow-server:5000/#/experiments/16
📝 Registrando modelo en MLflow: RandomForestClassifier


2025/09/12 17:53:01 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run RandomForestClassifier at: http://mlflow-server:5000/#/experiments/16/runs/0c783c8be5b0439583a410599aae1852
🧪 View experiment at: http://mlflow-server:5000/#/experiments/16
📝 Registrando modelo en MLflow: XGBClassifier


2025/09/12 17:53:06 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run XGBClassifier at: http://mlflow-server:5000/#/experiments/16/runs/b099ad08b4db40aa8cc8c6402237d16d
🧪 View experiment at: http://mlflow-server:5000/#/experiments/16
📝 Registrando modelo en MLflow: SVC


2025/09/12 17:53:11 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run SVC at: http://mlflow-server:5000/#/experiments/16/runs/bb687417c8314480b35b1472d9875f69
🧪 View experiment at: http://mlflow-server:5000/#/experiments/16
